## data_preparation_colab
- Role: prepare the raw file and create clean train/test files.
- I kept this notebook short and direct.


In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

raw_path = Path('disease_symptoms_raw.csv')
if not raw_path.exists():
    raw_path = Path('data/raw/disease_symptoms_raw.csv')

print('raw_path:', raw_path)
raw_df = pd.read_csv(raw_path)
raw_df.head()


### Clean columns
- I clean the column names before splitting the file.


In [ ]:
def clean_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = df.columns.astype(str).str.strip()
    drop_cols = [c for c in df.columns if c == '' or c.lower().startswith('unnamed')]
    if drop_cols:
        df = df.drop(columns=drop_cols)
    return df

raw_df = clean_columns(raw_df)
target = 'prognosis'
symptom_cols = [c for c in raw_df.columns if c != target]
print('Raw shape:', raw_df.shape)


### Split raw data
- I split by unique symptom pattern so the same pattern does not appear in both train and test.


In [ ]:
from sklearn.model_selection import train_test_split

raw_df['_symptom_signature'] = raw_df[symptom_cols].astype(str).agg('|'.join, axis=1)
unique_patterns = raw_df[[target, '_symptom_signature']].drop_duplicates()

test_signatures = []
for label, group in unique_patterns.groupby(target):
    _, label_test = train_test_split(
        group['_symptom_signature'],
        test_size=0.2,
        random_state=42,
    )
    test_signatures.extend(label_test.tolist())

test_signature_set = set(test_signatures)
train_df = raw_df[~raw_df['_symptom_signature'].isin(test_signature_set)].copy()
test_df = raw_df[raw_df['_symptom_signature'].isin(test_signature_set)].copy()

output_dir = Path('data')
output_dir.mkdir(exist_ok=True)
train_output = output_dir / 'Training.csv'
test_output = output_dir / 'Testing.csv'

train_df = train_df.drop(columns=['_symptom_signature'])
test_df = test_df.drop(columns=['_symptom_signature'])
train_df.to_csv(train_output, index=False)
test_df.to_csv(test_output, index=False)

print('Train shape:', train_df.shape)
print('Test shape:', test_df.shape)
print('Saved:', train_output)
print('Saved:', test_output)


### Leakage check
- I check that train and test do not share the same symptom pattern.


In [ ]:
train_signatures = set(train_df[symptom_cols].astype(str).agg('|'.join, axis=1))
test_signatures = set(test_df[symptom_cols].astype(str).agg('|'.join, axis=1))
print('Shared symptom patterns:', len(train_signatures & test_signatures))


### Raw dataset check
- I checked the size, number of diseases, and sample rows.


In [ ]:
print('Rows and columns:', raw_df.shape)
print('Number of diseases:', raw_df[target].nunique())
print('First 10 columns:', list(raw_df.columns[:10]))
raw_df.head()


### Split summary
- I checked the new train and test files after saving them.


In [ ]:
print('Train rows:', len(train_df))
print('Test rows:', len(test_df))
print('Train unique diseases:', train_df[target].nunique())
print('Test unique diseases:', test_df[target].nunique())
display(train_df[target].value_counts().head())


### Short findings
- The raw file has 4,962 rows and 41 diseases.
- I removed the extra empty column before working on the data.
- The notebook saves new `data/Training.csv` and `data/Testing.csv` files.
- Train and test share 0 symptom patterns.
